# FluHost-AA · F1.1 First Kaggle Smoke Run

**Purpose:** verify that the frozen PB2 pipeline runs end-to-end on a tiny public-data fixture.

This 10-sequence fixture is for software validation only. It is not large enough for scientific inference and is not expected to reproduce the dissertation's full-cohort frequencies.


In [ ]:
from pathlib import Path
import sys, json, platform
import pandas as pd

WORK = Path("/kaggle/working/fluhost_f1_1")
if not Path("/kaggle/working").exists():
    WORK = Path("/mnt/data/fluhost_f1_kaggle_pb2/f1_1_notebook_work")
WORK.mkdir(parents=True, exist_ok=True)

candidates = []
kaggle_input = Path("/kaggle/input")
if kaggle_input.exists():
    for d in kaggle_input.iterdir():
        if d.is_dir() and all((d/f).exists() for f in [
            "metadata.tsv", "pb2_aligned.fasta", "fluhost_pb2.py"
        ]):
            candidates.append(d)

# Local execution fallback used only to validate the notebook before Kaggle upload.
if not candidates:
    local = Path("/mnt/data/fluhost_f1_kaggle_pb2/f1_1_smoke_dataset")
    if local.exists() and all((local/f).exists() for f in [
        "metadata.tsv", "pb2_aligned.fasta", "fluhost_pb2.py"
    ]):
        candidates = [local]

if len(candidates) != 1:
    raise RuntimeError(
        f"Expected exactly one FluHost F1.1 dataset; found {len(candidates)}: {candidates}"
    )

DATA = candidates[0]
META_PATH = DATA/"metadata.tsv"
FASTA_PATH = DATA/"pb2_aligned.fasta"
sys.path.insert(0, str(DATA))

print("Dataset mount:", DATA)
print("Python:", platform.python_version())


In [ ]:
from fluhost_pb2 import (
    PB2Config, load_metadata, load_fasta, intersect_inputs,
    aa_frequency_table, add_association_stats, requested_residue_row,
    sha256_file, write_manifest
)

metadata = load_metadata(META_PATH)
seqs = load_fasta(FASTA_PATH)
metadata, seqs, accounting = intersect_inputs(metadata, seqs)

accounting["matched_avian"] = int((metadata.host_group == "Avian").sum())
accounting["matched_human"] = int((metadata.host_group == "Human").sum())
print(json.dumps(accounting, indent=2))
(WORK/"input_accounting.json").write_text(json.dumps(accounting, indent=2), encoding="utf-8")


In [ ]:
cfg = PB2Config(exclude_gap=True, exclude_x=True)
freq = aa_frequency_table(metadata, seqs, cfg)
assoc = add_association_stats(freq)

freq.to_csv(WORK/"pb2_frequency.csv", index=False)
assoc.to_csv(WORK/"pb2_association.csv", index=False)

print("Alignment length:", len(next(iter(seqs.values()))))
print("Association rows:", len(assoc))
display(assoc.head())


## Predefined dissertation anchor check

The dissertation's PB2 screening includes **E65D** and reports a historical full-cohort frequency for residue D at position 65. For this tiny smoke fixture, absence of D is reported explicitly as zero rather than treated as an error. This is an engineering check, not a reproduction-quality result.


In [ ]:
PAPER = {
    "position": 65,
    "aa": "D",
    "avian_frequency": 0.0395,
    "human_frequency": 0.3626,
}

anchor = requested_residue_row(assoc, position=PAPER["position"], aa=PAPER["aa"])
anchor["paper_avian_frequency"] = PAPER["avian_frequency"]
anchor["paper_human_frequency"] = PAPER["human_frequency"]
anchor["avian_abs_pp_deviation"] = (anchor["avian_frequency"] - PAPER["avian_frequency"]).abs() * 100
anchor["human_abs_pp_deviation"] = (anchor["human_frequency"] - PAPER["human_frequency"]).abs() * 100
anchor.to_csv(WORK/"pb2_65D_anchor.csv", index=False)
display(anchor)


In [ ]:
manifest = {
    "project": "FluHost-AA",
    "phase": "F1.1",
    "run_type": "engineering smoke test",
    "records": int(len(metadata)),
    "avian_n": int((metadata.host_group == "Avian").sum()),
    "human_n": int((metadata.host_group == "Human").sum()),
    "metadata_sha256": sha256_file(META_PATH),
    "fasta_sha256": sha256_file(FASTA_PATH),
    "exclude_gap": cfg.exclude_gap,
    "exclude_x": cfg.exclude_x,
    "paper_anchor": PAPER,
    "python": platform.python_version(),
    "pandas": pd.__version__,
}
write_manifest(WORK/"run_manifest.json", manifest)

print("F1.1 SMOKE RUN: PASS")
print(json.dumps(manifest, indent=2))
print("\nOutputs:")
for p in sorted(WORK.iterdir()):
    print(" -", p.name)
